## 1. Setup


In [ ]:
import csv
import io
import json
import shutil
import zipfile
from collections import Counter
from pathlib import Path

INPUT = Path("/kaggle/input") if Path("/kaggle/input").is_dir() else Path.cwd()
OUTPUT = (Path("/kaggle/working") if Path("/kaggle/working").is_dir() else Path.cwd()) / "TMFD_single_face.zip"
SPLIT_GAP = 10
REJECTED_IDS = set()
PREFIX = "TMFD_single_face/"


## 2. Source data


In [ ]:
roots = sorted(path.parent for path in INPUT.rglob("images")
               if (path.parent / "labels").is_dir() and (path / "easy/t").is_dir())
archive = None
if roots:
    root = roots[0]
    label_paths = [path.relative_to(root).as_posix() for path in (root / "labels").rglob("*.txt")]
    readme = next((path for path in [root / "README.md", root.parent / "README.md"] if path.is_file()), None)
else:
    archives = sorted(INPUT.rglob("TMFD Dataset.zip"))
    assert archives, "Attach TMFD Dataset.zip or place it beside this notebook."
    archive = zipfile.ZipFile(archives[0])
    names = set(archive.namelist())
    anchor = "images/easy/d/easy_d_1.png"
    prefix = next(name[:-len(anchor)] for name in names if name.endswith(anchor))
    label_paths = [name[len(prefix):] for name in names
                   if name.startswith(prefix + "labels/") and name.endswith(".txt")]
    readme = next((name for name in sorted(names) if Path(name).name.lower() == "readme.md"), None)


def open_source(relative):
    return archive.open(prefix + relative) if archive is not None else (root / relative).open("rb")


def read_source(relative):
    with open_source(relative) as stream:
        return stream.read()


def exists(relative):
    return prefix + relative in names if archive is not None else (root / relative).is_file()


## 3. Single-face frames and splits


In [ ]:
frames = []
limits = {}
for path in label_paths:
    coordinates = list(map(int, read_source(path).decode("utf-8-sig").split()))
    assert len(coordinates) % 4 == 0, path
    scene, frame = Path(path).stem.rsplit("_", 1)
    frame = int(frame)
    limits[scene] = max(limits.get(scene, 0), frame)
    frames.append({"id": f"{scene}_{frame}", "scene": scene, "frame": frame,
                   "faces": len(coordinates) // 4, "label": path})


def assign_split(row):
    train_end, validation_end = int(limits[row["scene"]] * 0.7), int(limits[row["scene"]] * 0.85)
    if row["frame"] <= train_end - SPLIT_GAP:
        return "train"
    if train_end + SPLIT_GAP < row["frame"] <= validation_end - SPLIT_GAP:
        return "validation"
    if row["frame"] > validation_end + SPLIT_GAP:
        return "test"
    return "gap"


single = [dict(row, split=assign_split(row)) for row in frames if row["faces"] == 1]
selected = sorted((row for row in single if row["split"] != "gap" and row["id"] not in REJECTED_IDS),
                  key=lambda row: (row["scene"], row["frame"]))
assert selected and {row["split"] for row in selected} == {"train", "validation", "test"}
assert len({row["id"] for row in selected}) == len(selected)
files = []
for row in selected:
    scene, frame = row["scene"], row["frame"]
    files.extend([f"images/{scene}/rgb/{scene}_rgb_{frame}.png",
                  f"images/{scene}/d/{scene}_d_{frame}.png", row["label"]])
assert len(files) == len(set(files)) and all(exists(path) for path in files), "Missing or duplicate source files."
print({"source_frames": len(frames), "single_face_frames": len(single), "exported_frames": len(selected)})
print(dict(Counter(row["split"] for row in selected)))


## 4. Export ZIP


In [ ]:
manifest = io.StringIO()
writer = csv.DictWriter(manifest, fieldnames=["id", "scene", "frame", "split"], extrasaction="ignore")
writer.writeheader()
writer.writerows(selected)
metadata = {
    "dataset": "TMFD_single_face", "schema_version": 1,
    "source": "https://github.com/wiktormucha/tmfd_dataset",
    "source_frames": len(frames), "single_face_frames": len(single), "selected_frames": len(selected),
    "scene_frame_max": limits, "split_gap": SPLIT_GAP, "split": "filename_blocks_with_gaps",
    "rejected_ids": sorted(REJECTED_IDS), "image_shape": [480, 640],
    "images_unchanged": True, "selection": "single_face_excluding_split_gaps",
    "depth_quality_filter": "performed_in_training_notebook",
}
expected = {PREFIX + path for path in files} | {PREFIX + "manifest.csv", PREFIX + "subset.json"}
with zipfile.ZipFile(OUTPUT, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=1) as target:
    for index, relative in enumerate(files, 1):
        with open_source(relative) as source, target.open(PREFIX + relative, "w") as destination:
            shutil.copyfileobj(source, destination, length=1024 * 1024)
        if index % 500 == 0 or index == len(files):
            print(f"Copied {index}/{len(files)} files", flush=True)
    target.writestr(PREFIX + "manifest.csv", manifest.getvalue())
    target.writestr(PREFIX + "subset.json", json.dumps(metadata, ensure_ascii=False, indent=2))
    if readme is not None:
        content = archive.read(readme) if archive is not None else readme.read_bytes()
        target.writestr(PREFIX + "README.md", content)
        expected.add(PREFIX + "README.md")
print(OUTPUT)
print(f"{OUTPUT.stat().st_size / 1024 ** 2:.1f} MiB")


## 5. Verify and download


In [ ]:
with zipfile.ZipFile(OUTPUT) as subset:
    assert set(subset.namelist()) == expected
    assert subset.testzip() is None, "ZIP integrity check failed."
    saved = list(csv.DictReader(io.StringIO(subset.read(PREFIX + "manifest.csv").decode())))
    assert len(saved) == len(selected)
    assert {row["id"]: row["split"] for row in saved} == {row["id"]: row["split"] for row in selected}
    if archive is not None:
        assert all(subset.getinfo(PREFIX + relative).CRC == archive.getinfo(prefix + relative).CRC
                   for relative in files)
if archive is not None:
    archive.close()
print("ZIP verified. RGB and depth frames retain their original resolution and bytes.")
print(OUTPUT)
